# 01 — Data Quality & Understanding

**Project:** E-Commerce Delivery Performance & Customer Satisfaction  
**Dataset:** Olist Brazilian E-Commerce Public Dataset  
**Unit of analysis:** One order (`order_id`)

## Objective
Understand the order table before calculating delivery metrics. We will inspect its structure, validate order identifiers, quantify missing values, and investigate why delivery timestamps are missing.

**Important:** Missing values are not automatically errors. Keep the raw DataFrame unchanged and create working copies for analysis.

Run this notebook from the repository root. Place `olist_orders_dataset.csv` in `data/raw/`.

## 0. Setup and data loading

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

sns.set_theme(style="whitegrid")

# Try common locations used in Jupyter and Google Colab.
candidates = [
    Path("/data/raw/olist_orders_dataset.csv"),
    Path("dataset/olist_orders_dataset.csv"),
    Path("olist_orders_dataset.csv"),
    Path("/content/olist_orders_dataset.csv"),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Place olist_orders_dataset.csv beside this notebook, in dataset/, "
        "or in /content/ (Colab), then rerun this cell."
    )

# Raw source data: preserve this DataFrame without modifying it.
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {DATA_PATH}")

Loaded 99,441 rows and 8 columns from dataset/olist_orders_dataset.csv


## 1. Challenge 01 — Initial data inspection

**Business question:** What information is available, what does one row represent, and are the timestamp fields ready for time calculations?

In [7]:
display(df.head())
print("Shape (rows, columns):", df.shape)
print("\nColumn data types:")
print(df.dtypes)
print("\nMissing values by column:")
display(df.isna().sum().rename("missing_count").to_frame())
print("\nUnique order IDs:", df["order_id"].nunique())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


Shape (rows, columns): (99441, 8)

Column data types:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Missing values by column:


,missing_count
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0



Unique order IDs: 99441


### Interpretation — Challenge 01

- Each row is intended to represent an order record. Verify this by checking whether `order_id` is unique; do not assume uniqueness without testing it.
- The timestamp columns are initially imported as strings/objects. Convert them to datetime before subtracting dates or extracting calendar components.
- Missing delivery timestamps are not automatically data errors. Orders that are cancelled, shipped, or otherwise not completed may not have an actual customer-delivery timestamp.
- The order table supports order-level delivery analysis. It does not, by itself, contain the customer review score; that requires a separate review table later.

## 2. Challenge 02 — Data quality audit

**Business question:** Are order identifiers duplicated, and where are missing values concentrated?

In [8]:
# Duplicate IDs can cause double-counting in order-level metrics.
duplicate_count = df.duplicated(subset=["order_id"]).sum()
print(f"Duplicate order IDs: {duplicate_count:,}")

# Missing-value percentages here use ALL orders as the denominator.
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct_all_orders": df.isna().mean().mul(100),
}).sort_values("missing_pct_all_orders", ascending=False)
display(missing_summary.round(3))

# A specific consistency check: delivered status but no actual customer-delivery timestamp.
delivered_missing_date = df.loc[
    df["order_status"].eq("delivered") & df["order_delivered_customer_date"].isna()
].copy()
print(f"Delivered-status orders missing actual delivery timestamp: {len(delivered_missing_date):,}")
display(delivered_missing_date.head())

Duplicate order IDs: 0


,missing_count,missing_pct_all_orders
order_delivered_customer_date,2965,2.982
order_delivered_carrier_date,1783,1.793
order_approved_at,160,0.161
order_id,0,0.000
order_purchase_timestamp,0,0.000
order_status,0,0.000
customer_id,0,0.000
order_estimated_delivery_date,0,0.000


Delivered-status orders missing actual delivery timestamp: 8


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00


### Interpretation — Challenge 02

In the earlier run of this notebook:

- The table contained **99,441 orders** and **zero duplicate `order_id` values**.
- **2,965 orders (2.982% of all orders)** had a missing actual customer-delivery timestamp.
- **Eight** records had `order_status == "delivered"` but no actual customer-delivery timestamp. These are worth flagging for investigation; do not silently delete or impute them.

**Why not fill actual delivery dates with estimated dates?** The estimated date is a target/promise, not an observed delivery event. Filling actual dates with estimates would manufacture on-time deliveries and bias punctuality analysis.

## 3. Challenge 03 — Understand missing delivery timestamps

**Business question:** Are missing actual delivery dates mainly associated with orders that have not reached the delivered status?

The first percentage below answers: “Of all records with a missing actual delivery date, what share belongs to each status?” Its denominator is the **missing-date subset**, not all orders.

In [9]:
missing_delivery = df.loc[df["order_delivered_customer_date"].isna()].copy()
missing_by_status = (
    missing_delivery.groupby("order_status")
    .size()
    .rename("missing_count")
    .to_frame()
    .sort_values("missing_count", ascending=False)
)
missing_by_status["share_of_missing_pct"] = (
    missing_by_status["missing_count"] / len(missing_delivery) * 100
)
display(missing_by_status.round(2))

# A different question: among ALL delivered-status orders, what share lacks an actual date?
delivered_total = df["order_status"].eq("delivered").sum()
missing_delivered_count = len(delivered_missing_date)
missing_among_delivered_pct = missing_delivered_count / delivered_total * 100
print(
    f"Missing actual delivery timestamp among delivered-status orders: "
    f"{missing_delivered_count:,}/{delivered_total:,} ({missing_among_delivered_pct:.4f}%)"
)

,missing_count,share_of_missing_pct
order_status,,
shipped,1107,37.34
canceled,619,20.88
unavailable,609,20.54
invoiced,314,10.59
processing,301,10.15
delivered,8,0.27
created,5,0.17
approved,2,0.07


Missing actual delivery timestamp among delivered-status orders: 8/96,478 (0.0083%)


### Interpretation — Challenge 03

The earlier run found **1,107 shipped orders**, representing approximately **37.34% of all orders with missing actual delivery dates**. That is a share of the missing-date group; it is not the percentage of all shipped orders.

There were also **619 cancelled**, **609 unavailable**, and smaller numbers of other statuses with missing delivery timestamps. This pattern suggests that many missing values are structurally understandable because those orders may not have reached customer delivery.

Only **8 delivered-status orders** lacked the actual delivery timestamp. The earlier run calculated this as approximately **0.0083% of delivered-status orders**. Keep the denominator explicit whenever reporting a percentage.

**Analyst decision:** Preserve the records, flag the eight inconsistent delivered-status rows, and exclude them only from metrics that require an actual delivery timestamp.